# ts_ols_and_nl_of_pre_autocorr

> 对应代码：`EconometricsCode/code_in_notes/Chap.27/ts_ols_and_nl_of_pre_autocorr.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.27`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.27")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

In [ ]:
%%stata
clear
set more off
set obs 2000
set seed 505

gen t=_n
tsset t

我们生成如下 DGP：误差项 u 服从 AR(1) 过程 $u_t=0.5u_{t-1}+e_t$；解释变量 x 是**前定变量**（predetermined），$x_t=2u_{t-1}+v_t$，即 x 可以与滞后的 u 相关、但与当期冲击 e 不相关；最后 $y_t=1+3x_t+u_t$。这对应教材中误差自相关且解释变量仅为前定变量的情形。

In [ ]:
%%stata
gen e=rnormal()
gen u=0
replace u=0.5*L.u+e if t>1
// x为前定变量，所以可以与滞后的u相关
gen x=0
replace x=2*L.u+rnormal() if t>1
gen y=1+3*x+u

先直接对 y 关于 x 做 OLS。由于 u 自相关且 x 与 L.u 相关，$x_t$ 与 $u_t$ 通过 $u_{t-1}$ 产生相关——严格外生性被破坏，OLS 不一致，我们将看到系数偏离真值 3。

In [ ]:
%%stata
// 简单的最小二乘法是错误的
reg y x

一种修正思路是把模型改写成动态完备的形式：由 $u_t=\rho u_{t-1}+e_t$ 可得 $y_t=\alpha(1-\rho)+\rho y_{t-1}+\beta x_t-\rho\beta x_{t-1}+e_t$，其中 $e_t$ 为白噪声且与各回归元无关，因此直接回归 `y L.y x L.x`（不施加参数约束）即得到一致的 OLS 估计。

In [ ]:
%%stata
// 更正的OLS
reg y L.y x L.x

另一种做法是用非线性最小二乘（NLS）直接估计带共同因子约束（common factor restriction）的原模型：在条件均值方程中显式写出 $\alpha(1-\rho)+\rho L.y+\beta x-\rho\beta L.x$，让 $\rho$ 同时出现在 L.y 与 L.x 的系数中（系数间满足非线性约束）。NLS 利用了误差 AR(1) 结构所蕴含的约束，因此比无约束 OLS 更有效率。

In [ ]:
%%stata
// 非线性最小二乘
nl (y={alpha}*(1-{rho})+{rho}*L.y+{beta}*x-{rho}*{beta}*L.x) if t>1